<a href="https://colab.research.google.com/github/Kenny625819/Applied-Data-Science/blob/main/IEEE_JBHI_submit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Unified Reproducible Analysis Script (Final 177-case version)
-------------------------------------------------------------
Multimodal Survival Prediction in Metastatic Spine Surgery

This script reproduces the main analyses:

    • 5-fold out-of-fold (OOF) LightGBM models
      with isotonic calibration for 3M / 6M / 12M survival

    • ROC curves and calibration plots (black-and-white final version)
    • SHAP summary plots
    • SHAP mean-absolute heatmap

    • Performance_Summary_OOF_177.xlsx
          - AUC, 95% CI, Sensitivity, Specificity, F1, Brier (LightGBM)
          - AUC, 95% CI, Sensitivity, Specificity, F1 (Tokuhashi, Katagiri)
          - DeLong-type p values for LightGBM vs Tokuhashi / Katagiri

    • Ablation_Study_Output_177.xlsx
          - Full model (clinical + ESCC) vs Clinical-only

    • Spearman_Correlation_177.xlsx
          - Spearman correlation between ESCC and ECOG / Barthel

Input:
    MRI_ALLdata_OOF177.xlsx

Outputs:
    ./RESULTS_FIGURES_177/
"""

# =============================================================================
# Imports
# =============================================================================
import numpy as np
import pandas as pd
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.stats import norm, spearmanr

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    brier_score_loss,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.isotonic import IsotonicRegression

# =============================================================================
# Global settings
# =============================================================================
OUT_DIR = Path("RESULTS_FIGURES_177")
OUT_DIR.mkdir(exist_ok=True)

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False

# =============================================================================
# Utilities
# =============================================================================
def map_sex(x):
    s = str(x).strip().lower()
    if s in ["1", "m", "male", "男", "男性"]:
        return 1
    if s in ["0", "f", "female", "女", "女性"]:
        return 0
    return np.nan


def map_escc(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower().replace(" ", "")
    return {"1a": 1, "1b": 2, "1c": 3, "2": 4, "3": 5}.get(s, np.nan)


def frankel_bin(x):
    s = str(x).strip().upper()
    if s in ["A", "B", "C"]:
        return 0
    if s in ["D", "E"]:
        return 1
    return np.nan


def map_yesno(x):
    s = str(x).strip().lower()
    if s in ["yes", "y", "true", "1", "あり", "有"]:
        return 1
    if s in ["no", "n", "false", "0", "なし", "無"]:
        return 0
    return np.nan


def auc_ci_bootstrap(y, p, n_boot=2000, seed=42):
    y = np.asarray(y)
    p = np.asarray(p)
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y))
    aucs = []

    for _ in range(n_boot):
        b = rng.choice(idx, len(idx), replace=True)
        try:
            aucs.append(roc_auc_score(y[b], p[b]))
        except ValueError:
            pass

    auc = roc_auc_score(y, p)
    lo, hi = np.percentile(aucs, 2.5), np.percentile(aucs, 97.5)
    return auc, lo, hi


def delong_type_test(y, p1, p2):
    """
    DeLong-type paired ROC comparison.
    This is a simplified paired comparison, not the full canonical DeLong covariance estimator.
    Returns:
        delta_auc, p_value
    """
    y = np.asarray(y)
    p1 = np.asarray(p1)
    p2 = np.asarray(p2)

    m = int(np.sum(y))
    n = len(y) - m

    if m == 0 or n == 0:
        return np.nan, np.nan

    a1 = roc_auc_score(y, p1)
    a2 = roc_auc_score(y, p2)

    var = (a1 * (1 - a1) / max(n, 1)) + (a2 * (1 - a2) / max(n, 1))
    z = (a1 - a2) / np.sqrt(var + 1e-12)
    p = 2 * (1 - norm.cdf(abs(z)))
    return a1 - a2, p


def get_binary_metrics(y, p, thr=0.5):
    y = np.asarray(y)
    p = np.asarray(p)
    yhat = (p >= thr).astype(int)

    cm = confusion_matrix(y, yhat, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    _, _, f1, _ = precision_recall_fscore_support(
        y, yhat, average="binary", zero_division=0
    )
    return sens, spec, f1


def youden_threshold(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    return thr[np.argmax(tpr - fpr)]


# =============================================================================
# OOF LightGBM + isotonic calibration
# =============================================================================
def run_lgb_oof_calibrated(X, y):
    """
    Train LightGBM using OOF predictions and isotonic calibration.
    Returns:
        calibrated_oof_predictions, final_model
    """
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    params = dict(
        objective="binary",
        metric="auc",
        learning_rate=0.05,
        num_leaves=31,
        n_estimators=500,
        class_weight="balanced",
        random_state=42,
    )

    oof = np.zeros(len(y))

    for tr, te in skf.split(X, y):
        model = lgb.LGBMClassifier(**params)
        model.fit(X.iloc[tr], y[tr])
        oof[te] = model.predict_proba(X.iloc[te])[:, 1]

    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(oof, y)
    calibrated = iso.transform(oof)

    final_model = lgb.LGBMClassifier(**params)
    final_model.fit(X, y)

    return calibrated, final_model


# =============================================================================
# Plotting
# =============================================================================
def plot_roc(ax, y, p_lgb, p_tok, p_kat, title):
    fpr_lgb, tpr_lgb, _ = roc_curve(y, p_lgb)
    fpr_tok, tpr_tok, _ = roc_curve(y, p_tok)
    fpr_kat, tpr_kat, _ = roc_curve(y, p_kat)

    ax.plot(
        fpr_lgb, tpr_lgb,
        color="black", lw=2.5,
        label=f"LightGBM model (AUC={roc_auc_score(y, p_lgb):.3f})"
    )
    ax.plot(
        fpr_tok, tpr_tok,
        color="black", ls="--", lw=2.0,
        label=f"Revised Tokuhashi (AUC={roc_auc_score(y, p_tok):.3f})"
    )
    ax.plot(
        fpr_kat, tpr_kat,
        color="black", ls=":", lw=2.0,
        label=f"New Katagiri (AUC={roc_auc_score(y, p_kat):.3f})"
    )

    ax.plot([0, 1], [0, 1], "--", color="gray", lw=1.2)

    ax.set_title(title, fontsize=18)
    ax.set_xlabel("1 - Specificity", fontsize=18)
    ax.set_ylabel("Sensitivity", fontsize=18)
    ax.tick_params(labelsize=16)
    ax.legend(loc="lower right", fontsize=12)


def plot_calibration(ax, y, p):
    d = pd.DataFrame({"y": y, "p": p})
    d["bin"] = pd.qcut(d["p"], q=10, duplicates="drop")
    g = d.groupby("bin", observed=False).agg(
        obs=("y", "mean"),
        pred=("p", "mean")
    ).reset_index()

    ax.plot([0, 1], [0, 1], "--", color="gray", lw=1.2)
    ax.plot(g["pred"], g["obs"], "o-", color="black", lw=2)

    ax.set_title("Calibration plot", fontsize=18)
    ax.set_xlabel("Predicted probability", fontsize=18)
    ax.set_ylabel("Observed frequency", fontsize=18)
    ax.tick_params(labelsize=16)


def shap_summary_plot(model, X, title, filename, rename_map=None):
    X_disp = X.copy()
    if rename_map is not None:
        X_disp = X_disp.rename(columns=rename_map)

    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X)

    if isinstance(sv, list):
        sv = sv[1]

    shap.summary_plot(sv, X_disp, show=False)

    fig = plt.gcf()
    ax = plt.gca()

    ax.tick_params(axis="x", labelsize=12)
    for t in ax.get_yticklabels():
        t.set_fontsize(12)

    try:
        cbar = fig.axes[-1]
        cbar.tick_params(labelsize=12)
        cbar.set_ylabel(cbar.get_ylabel(), fontsize=12)
    except Exception:
        pass

    plt.title(title, fontsize=16)
    plt.tight_layout()
    plt.savefig(OUT_DIR / filename, dpi=600, bbox_inches="tight")
    plt.close()


def shap_mean_heatmap(models, X_dict, features, filename, rename_map=None):
    display_features = [rename_map.get(f, f) if rename_map else f for f in features]
    table = pd.DataFrame(index=display_features, columns=models.keys())

    for tag, model in models.items():
        X = X_dict[tag]
        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(X)

        if isinstance(sv, list):
            sv = sv[1]

        abs_mean = np.abs(sv).mean(axis=0)
        for f_orig, f_disp, val in zip(features, display_features, abs_mean):
            table.loc[f_disp, tag] = val

    if "3M" in models:
        table = table.sort_values(by="3M", ascending=False)

    plt.figure(figsize=(8, 7))
    ax = plt.gca()
    im = ax.imshow(table.astype(float).values, cmap="Greys", aspect="auto")

    ax.set_xticks(np.arange(len(table.columns)))
    ax.set_xticklabels(table.columns, fontsize=12)
    ax.set_yticks(np.arange(len(table.index)))
    ax.set_yticklabels(table.index, fontsize=12)

    for i in range(table.shape[0]):
        for j in range(table.shape[1]):
            val = float(table.iloc[i, j])
            color = "white" if val > np.nanmax(table.astype(float).values) * 0.5 else "black"
            ax.text(j, i, f"{val:.3f}", ha="center", va="center", color=color, fontsize=11)

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("mean(|SHAP value|)", fontsize=12)
    cbar.ax.tick_params(labelsize=12)

    ax.set_title("SHAP Heatmap", fontsize=16)
    plt.tight_layout()
    plt.savefig(OUT_DIR / filename, dpi=600, bbox_inches="tight")
    plt.close()


# =============================================================================
# Load data
# =============================================================================
INPUT_XLSX = "MRI_ALLdata_OOF177.xlsx"
df_raw = pd.read_excel(INPUT_XLSX)
df_raw.columns = df_raw.columns.str.strip()

df = pd.DataFrame({
    "Age": pd.to_numeric(df_raw["Age"], errors="coerce"),
    "Sex": df_raw["Sex"].apply(map_sex),
    "Number of spinal metastases": pd.to_numeric(df_raw["Number of Spinal Metastases"], errors="coerce"),
    "Albumin": pd.to_numeric(df_raw["Serum Albumin"], errors="coerce"),
    "CRP": pd.to_numeric(df_raw["CRP"], errors="coerce"),
    "ESCC": df_raw["ESCC"].apply(map_escc),
    "ECOG": pd.to_numeric(df_raw["Performance Status (ECOG)"], errors="coerce"),
    "Frankel grade": df_raw["Frankel Grade"].apply(frankel_bin),
    "Barthel Index": pd.to_numeric(df_raw["Barthel Index (ADL)"], errors="coerce"),
    "Malignancy (New Katagiri score)": pd.to_numeric(df_raw["Malignancy (Katagiri Score)"], errors="coerce"),
    "Visceral metastasis": df_raw["Visceral Metastasis (Yes=1/No=0)"].apply(map_yesno),
    "BMI": pd.to_numeric(df_raw["Body Mass Index (BMI)"], errors="coerce"),

    # Classical prognostic systems
    "Tokuhashi_binary": (pd.to_numeric(df_raw["Revised Tokuhashi score"], errors="coerce") >= 9).astype(float),
    "Katagiri_binary": (pd.to_numeric(df_raw["New Katagiri score"], errors="coerce") < 7).astype(float),

    # Outcomes
    "Y_3M": pd.to_numeric(df_raw["3-Month Survival (0=Death, 1=Alive)"], errors="coerce"),
    "Y_6M": pd.to_numeric(df_raw["6-Month Survival (0=Death, 1=Alive)"], errors="coerce"),
    "Y_12M": pd.to_numeric(df_raw["12-Month Survival (0=Death, 1=Alive)"], errors="coerce"),
})

# =============================================================================
# Feature definitions
# =============================================================================
FEATURES = [
    "CRP",
    "Malignancy (New Katagiri score)",
    "Age",
    "Barthel Index",
    "Albumin",
    "BMI",
    "ESCC",
    "ECOG",
    "Number of spinal metastases",
    "Sex",
    "Frankel grade",
    "Visceral metastasis",
]

FEATURE_RENAME = {
    "CRP": "CRP",
    "Malignancy (New Katagiri score)": "Malignancy (New Katagiri score)",
    "Age": "Age",
    "Barthel Index": "Barthel Index",
    "Albumin": "Albumin",
    "BMI": "BMI",
    "ESCC": "ESCC grade",
    "ECOG": "ECOG performance status",
    "Number of spinal metastases": "Number of spinal metastases",
    "Sex": "Sex",
    "Frankel grade": "Frankel grade",
    "Visceral metastasis": "Visceral metastasis",
}

FEATURES_NO_ESCC = [f for f in FEATURES if f != "ESCC"]

# =============================================================================
# 1. Spearman correlation
# =============================================================================
corr_sub = df.dropna(subset=["ESCC", "ECOG", "Barthel Index"]).copy()

rho_ecog, p_ecog = spearmanr(corr_sub["ESCC"], corr_sub["ECOG"])
rho_barthel, p_barthel = spearmanr(corr_sub["ESCC"], corr_sub["Barthel Index"])

corr_df = pd.DataFrame([
    {"Variable_1": "ESCC", "Variable_2": "ECOG", "Spearman_rho": rho_ecog, "p_value": p_ecog, "n": len(corr_sub)},
    {"Variable_1": "ESCC", "Variable_2": "Barthel Index", "Spearman_rho": rho_barthel, "p_value": p_barthel, "n": len(corr_sub)},
])

corr_df.to_excel(OUT_DIR / "Spearman_Correlation_177.xlsx", index=False)

print("\n=== Spearman correlation ===")
print(corr_df)

# =============================================================================
# 2. Main survival analysis
# =============================================================================
perf_records = []
models_by_horizon = {}
X_dict = {}

for tag, ycol in [("3M", "Y_3M"), ("6M", "Y_6M"), ("12M", "Y_12M")]:
    sub = df.dropna(subset=FEATURES + [ycol]).copy()
    X = sub[FEATURES]
    y = sub[ycol].astype(int).values

    X_dict[tag] = X

    # LightGBM OOF calibrated model
    p_lgb, model_lgb = run_lgb_oof_calibrated(X, y)
    models_by_horizon[tag] = model_lgb

    # Classical scores
    p_tok = sub["Tokuhashi_binary"].astype(float).values
    p_kat = sub["Katagiri_binary"].astype(float).values

    # Figures
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    plot_roc(axes[0], y, p_lgb, p_tok, p_kat, f"{tag} survival")
    plot_calibration(axes[1], y, p_lgb)
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"Figure3_{tag}_bw.png", dpi=600, bbox_inches="tight")
    plt.close()

    # SHAP summary
    shap_summary_plot(
        model_lgb, X,
        f"SHAP summary ({tag})",
        f"SHAP_{tag}_177.png",
        rename_map=FEATURE_RENAME
    )

    # Metrics with bootstrap CI
    lgb_auc, lgb_lo, lgb_hi = auc_ci_bootstrap(y, p_lgb)
    tok_auc, tok_lo, tok_hi = auc_ci_bootstrap(y, p_tok)
    kat_auc, kat_lo, kat_hi = auc_ci_bootstrap(y, p_kat)

    # DeLong-type tests
    delta_tok, p_lgb_vs_tok = delong_type_test(y, p_lgb, p_tok)
    delta_kat, p_lgb_vs_kat = delong_type_test(y, p_lgb, p_kat)

    # Threshold-based metrics
    thr_lgb = youden_threshold(y, p_lgb)
    sens_lgb, spec_lgb, f1_lgb = get_binary_metrics(y, p_lgb, thr_lgb)
    sens_tok, spec_tok, f1_tok = get_binary_metrics(y, p_tok, 0.5)
    sens_kat, spec_kat, f1_kat = get_binary_metrics(y, p_kat, 0.5)

    brier_lgb = brier_score_loss(y, p_lgb)

    perf_records.append({
        "Horizon": tag,
        "n": len(sub),

        "LightGBM AUC": lgb_auc,
        "LightGBM 95% CI low": lgb_lo,
        "LightGBM 95% CI high": lgb_hi,
        "LightGBM Sensitivity": sens_lgb,
        "LightGBM Specificity": spec_lgb,
        "LightGBM F1": f1_lgb,
        "LightGBM Brier": brier_lgb,

        "Tokuhashi AUC": tok_auc,
        "Tokuhashi 95% CI low": tok_lo,
        "Tokuhashi 95% CI high": tok_hi,
        "Tokuhashi Sensitivity": sens_tok,
        "Tokuhashi Specificity": spec_tok,
        "Tokuhashi F1": f1_tok,

        "Katagiri AUC": kat_auc,
        "Katagiri 95% CI low": kat_lo,
        "Katagiri 95% CI high": kat_hi,
        "Katagiri Sensitivity": sens_kat,
        "Katagiri Specificity": spec_kat,
        "Katagiri F1": f1_kat,

        "Delta AUC (LightGBM - Tokuhashi)": delta_tok,
        "p (LightGBM vs Tokuhashi)": p_lgb_vs_tok,
        "Delta AUC (LightGBM - Katagiri)": delta_kat,
        "p (LightGBM vs Katagiri)": p_lgb_vs_kat,
    })

perf_df = pd.DataFrame(perf_records)
perf_df.to_excel(OUT_DIR / "Performance_Summary_OOF_177.xlsx", index=False)

print("\n=== Performance summary ===")
print(perf_df)

# =============================================================================
# 3. SHAP mean heatmap
# =============================================================================
shap_mean_heatmap(
    models_by_horizon,
    X_dict,
    FEATURES,
    "SHAP_heatmap_3M_6M_12M_177_bw.png",
    rename_map=FEATURE_RENAME,
)

# =============================================================================
# 4. Ablation study
# =============================================================================
ablation_records = []

for tag, ycol in [("3M", "Y_3M"), ("6M", "Y_6M"), ("12M", "Y_12M")]:
    sub = df.dropna(subset=FEATURES + [ycol]).copy()

    X_full = sub[FEATURES]
    X_no = sub[FEATURES_NO_ESCC]
    y = sub[ycol].astype(int).values

    p_full, _ = run_lgb_oof_calibrated(X_full, y)
    p_no, _ = run_lgb_oof_calibrated(X_no, y)

    auc_full = roc_auc_score(y, p_full)
    auc_no = roc_auc_score(y, p_no)

    brier_full = brier_score_loss(y, p_full)
    brier_no = brier_score_loss(y, p_no)

    sens_full, spec_full, f1_full = get_binary_metrics(y, p_full, 0.5)
    sens_no, spec_no, f1_no = get_binary_metrics(y, p_no, 0.5)

    delta_auc, p_delong = delong_type_test(y, p_full, p_no)

    ablation_records.append({
        "Horizon": tag,
        "n": len(sub),
        "AUC_Full (clinical + ESCC)": auc_full,
        "AUC_No_ESCC (clinical only)": auc_no,
        "Delta AUC (Full - No_ESCC)": delta_auc,
        "p (DeLong-type)": p_delong,
        "Brier_Full": brier_full,
        "Brier_No_ESCC": brier_no,
        "Sensitivity_Full": sens_full,
        "Specificity_Full": spec_full,
        "F1_Full": f1_full,
        "Sensitivity_No_ESCC": sens_no,
        "Specificity_No_ESCC": spec_no,
        "F1_No_ESCC": f1_no,
    })

ablation_df = pd.DataFrame(ablation_records)
ablation_df.to_excel(OUT_DIR / "Ablation_Study_Output_177.xlsx", index=False)

print("\n=== Ablation study ===")
print(ablation_df)

print("\nAnalysis completed successfully.")
print("Results saved in:", OUT_DIR.resolve())



=== Spearman correlation ===
  Variable_1     Variable_2  Spearman_rho   p_value    n
0       ESCC           ECOG      0.072791  0.335631  177
1       ESCC  Barthel Index     -0.095055  0.208206  177
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 109, number of negative: 32
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000233 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 183
[LightGBM] [Info] Number of data points in the train set: 141, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM]

/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 83, number of negative: 58
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000057 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 183
[LightGBM] [Info] Number of data points in the train set: 141, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 61, number of negative: 80
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000055 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 184
[LightGBM] [Info] Number of data points in the train set: 141, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(



=== Performance summary ===
  Horizon    n  LightGBM AUC  LightGBM 95% CI low  LightGBM 95% CI high  \
0      3M  177      0.740693             0.663738              0.818036   
1      6M  177      0.814146             0.750127              0.871889   
2     12M  177      0.846558             0.788227              0.899090   

   LightGBM Sensitivity  LightGBM Specificity  LightGBM F1  LightGBM Brier  \
0              0.598540              0.825000     0.725664        0.149418   
1              0.778846              0.712329     0.786408        0.168078   
2              0.753247              0.810000     0.753247        0.152140   

   Tokuhashi AUC  ...  Katagiri AUC  Katagiri 95% CI low  \
0       0.608485  ...      0.665511             0.580959   
1       0.660103  ...      0.692966             0.625593   
2       0.698766  ...      0.702532             0.644662   

   Katagiri 95% CI high  Katagiri Sensitivity  Katagiri Specificity  \
0              0.750824              0.781022

/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


ストリーミング出力は最後の 5000 行に切り捨てられました。
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

In [2]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Supplementary Analysis:
Spearman Correlation and Integrated Partial Dependence Plot (PDP)
----------------------------------------------------------------

This script computes:

1. Spearman correlation between ESCC severity and
   clinical functional markers (ECOG and Barthel Index)
   for each survival horizon (3M, 6M, 12M).

2. Integrated partial dependence plot (PDP) for ESCC
   using trained LightGBM models.

Required variables from code5:
    - models_by_horizon
    - X_dict
    - OUT_DIR

Outputs:
    - Supplementary_ESCC_Spearman.xlsx
    - Supplementary_ESCC_PDP_all_bw.png
"""

from scipy.stats import spearmanr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


def compute_spearman_and_plot_pdp(models_by_horizon, X_dict, out_dir):
    """
    Compute Spearman correlations and generate integrated PDP plot.

    Parameters
    ----------
    models_by_horizon : dict
        Dictionary of trained LightGBM models for {"3M", "6M", "12M"}.
    X_dict : dict
        Dictionary of predictor matrices for each horizon.
    out_dir : pathlib.Path
        Output directory.
    """

    # ============================================================
    # 1. Spearman Correlation Summary
    # ============================================================
    spearman_records = []

    for tag in ["3M", "6M", "12M"]:
        X = X_dict[tag].copy()

        rho_ecog, p_ecog = spearmanr(X["ESCC"], X["ECOG"], nan_policy="omit")
        rho_barthel, p_barthel = spearmanr(X["ESCC"], X["Barthel Index"], nan_policy="omit")

        spearman_records.append({
            "Horizon": tag,
            "Spearman_ESCC_ECOG": rho_ecog,
            "p_value_ECOG": p_ecog,
            "Spearman_ESCC_Barthel": rho_barthel,
            "p_value_Barthel": p_barthel,
            "n": len(X),
        })

        print(f"\n=== Spearman ({tag}) ===")
        print(f"  ESCC vs ECOG    : rho={rho_ecog:.3f}, p={p_ecog:.4f}")
        print(f"  ESCC vs Barthel : rho={rho_barthel:.3f}, p={p_barthel:.4f}")

    spearman_df = pd.DataFrame(spearman_records)
    spearman_path = out_dir / "Supplementary_ESCC_Spearman.xlsx"
    spearman_df.to_excel(spearman_path, index=False)
    print(f"\nSaved Spearman correlation summary -> {spearman_path}")

    # ============================================================
    # 2. Integrated Partial Dependence Plot (PDP)
    # ============================================================
    escc_levels = sorted(X_dict["3M"]["ESCC"].dropna().unique())  # e.g. [2, 3, 4, 5]
    pdp = {}

    for tag in ["3M", "6M", "12M"]:
        model = models_by_horizon[tag]
        X_base = X_dict[tag].copy()

        vals = []
        for g in escc_levels:
            X_temp = X_base.copy()
            X_temp["ESCC"] = g
            prob_alive = model.predict_proba(X_temp)[:, 1]
            prob_death = 1 - prob_alive
            vals.append(np.mean(prob_death))

        pdp[tag] = np.array(vals)

    # Map ESCC numeric codes to Bilsky labels
    label_map = {2: "1b", 3: "1c", 4: "2", 5: "3"}
    tick_labels = [label_map.get(x, str(x)) for x in escc_levels]

    # ============================================================
    # 3. Plot Integrated PDP (black-and-white final version)
    # ============================================================
    plt.figure(figsize=(8, 6))

    plt.plot(
        escc_levels, pdp["3M"],
        color="black", lw=2.8, linestyle="-", marker="o",
        label="3M survival",
    )

    plt.plot(
        escc_levels, pdp["6M"],
        color="black", lw=2.4, linestyle="--", marker="o",
        label="6M survival",
    )

    plt.plot(
        escc_levels, pdp["12M"],
        color="black", lw=2.4, linestyle=":", marker="o",
        label="12M survival",
    )

    plt.title("Partial dependence of ESCC on predicted mortality risk", fontsize=18)
    plt.xlabel("ESCC grade", fontsize=16)
    plt.ylabel("Mean predicted mortality risk", fontsize=16)

    plt.xticks(escc_levels, tick_labels, fontsize=14)
    plt.yticks(fontsize=14)

    plt.legend(
        fontsize=12,
        frameon=True,
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        borderpad=0.8
    )

    plt.tight_layout()
    out_path = out_dir / "Supplementary_ESCC_PDP_all_bw.png"
    plt.savefig(out_path, dpi=600, bbox_inches="tight")
    plt.close()

    print(f"\nIntegrated PDP figure saved -> {out_path}")


# Execute
compute_spearman_and_plot_pdp(models_by_horizon, X_dict, OUT_DIR)



=== Spearman (3M) ===
  ESCC vs ECOG    : rho=0.073, p=0.3356
  ESCC vs Barthel : rho=-0.095, p=0.2082

=== Spearman (6M) ===
  ESCC vs ECOG    : rho=0.073, p=0.3356
  ESCC vs Barthel : rho=-0.095, p=0.2082

=== Spearman (12M) ===
  ESCC vs ECOG    : rho=0.073, p=0.3356
  ESCC vs Barthel : rho=-0.095, p=0.2082

Saved Spearman correlation summary -> RESULTS_FIGURES_177/Supplementary_ESCC_Spearman.xlsx

Integrated PDP figure saved -> RESULTS_FIGURES_177/Supplementary_ESCC_PDP_all_bw.png


In [3]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Final Table3 / Table4 unified analysis
--------------------------------------

This script generates:

1. Table3_ablation_177_final.xlsx
   - Full model (clinical + CNN-derived ESCC) vs Clinical-only
   - AUC, 95% CI, Brier
   - Delta AUC / Delta Brier
   - DeLong-type p value

2. Table4_expert_vs_cnn_177_final.xlsx
   - Clinical-only vs Clinical + Expert ESCC vs Clinical + CNN-derived ESCC
   - AUC, 95% CI, Brier
   - Delta AUC / Delta Brier (CNN vs Expert)
   - DeLong-type p value (CNN vs Expert)

Input:
    MRI_ALLdata_OOF177.xlsx
"""

# =============================================================================
# Imports
# =============================================================================
import numpy as np
import pandas as pd
import lightgbm as lgb

from scipy.stats import norm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.isotonic import IsotonicRegression


# =============================================================================
# Load data
# =============================================================================
INPUT_FILE = "MRI_ALLdata_OOF177.xlsx"

df = pd.read_excel(INPUT_FILE)
df = df.loc[:, ~df.columns.str.contains("^Unnamed")].copy()
df.columns = df.columns.str.strip()


# =============================================================================
# Utility mappings
# =============================================================================
def map_escc_any(x):
    """
    ESCC mapping:
        1a -> 1
        1b -> 2
        1c -> 3
        2  -> 4
        3  -> 5
    """
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x).strip().lower().replace(" ", "")
    return {"1a": 1, "1b": 2, "1c": 3, "2": 4, "3": 5}.get(s, np.nan)


def frankel_to_binary(x):
    s = str(x).strip().upper()
    if s in ["A", "B", "C"]:
        return 0
    if s in ["D", "E"]:
        return 1
    return np.nan


def map_sex_any(x):
    s = str(x).strip().lower()
    if s in ["1", "m", "male", "男", "男性"]:
        return 1
    if s in ["0", "f", "female", "女", "女性"]:
        return 0
    return np.nan


def map_yesno_any(x):
    s = str(x).strip().lower()
    if s in ["1", "yes", "y", "true", "あり", "有"]:
        return 1
    if s in ["0", "no", "n", "false", "なし", "無"]:
        return 0
    return np.nan


# =============================================================================
# Column preparation
# =============================================================================
# ESCC columns
df["ESCC_cnn"] = df["ESCC"].apply(map_escc_any)

if "Expert" in df.columns:
    df["ESCC_expert"] = df["Expert"].apply(map_escc_any)
else:
    raise ValueError("Column 'Expert' not found in MRI_ALLdata_OOF177.xlsx")

# Frankel
df["Frankel Grade"] = df["Frankel Grade"].apply(frankel_to_binary)

# Sex / visceral robust conversion if needed
if "Sex" in df.columns:
    df["Sex"] = df["Sex"].apply(map_sex_any)

if "Visceral Metastasis (Yes=1/No=0)" in df.columns:
    df["Visceral Metastasis (Yes=1/No=0)"] = df["Visceral Metastasis (Yes=1/No=0)"].apply(map_yesno_any)

# Force numeric conversion
NUMERIC_COLUMNS = [
    "CRP",
    "Malignancy (Katagiri Score)",
    "Age",
    "Barthel Index (ADL)",
    "Serum Albumin",
    "Body Mass Index (BMI)",
    "Performance Status (ECOG)",
    "Number of Spinal Metastases",
    "Sex",
    "Frankel Grade",
    "Visceral Metastasis (Yes=1/No=0)",
    "ESCC_cnn",
    "ESCC_expert",
    "3-Month Survival (0=Death, 1=Alive)",
    "6-Month Survival (0=Death, 1=Alive)",
    "12-Month Survival (0=Death, 1=Alive)",
]

for col in NUMERIC_COLUMNS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")


# =============================================================================
# Feature definitions
# =============================================================================
BASE_FEATURES = [
    "CRP",
    "Malignancy (Katagiri Score)",
    "Age",
    "Barthel Index (ADL)",
    "Serum Albumin",
    "Body Mass Index (BMI)",
    "Performance Status (ECOG)",
    "Number of Spinal Metastases",
    "Sex",
    "Frankel Grade",
    "Visceral Metastasis (Yes=1/No=0)",
]

FEATURES_FULL = BASE_FEATURES + ["ESCC_cnn"]
FEATURES_NO_ESCC = BASE_FEATURES

FEATURES_CLINICAL = BASE_FEATURES
FEATURES_EXPERT = BASE_FEATURES + ["ESCC_expert"]
FEATURES_CNN = BASE_FEATURES + ["ESCC_cnn"]


# =============================================================================
# Core statistics
# =============================================================================
def auc_ci_bootstrap(y, p, n_boot=2000, seed=42):
    y = np.asarray(y)
    p = np.asarray(p)

    rng = np.random.default_rng(seed)
    idx = np.arange(len(y))
    aucs = []

    for _ in range(n_boot):
        b = rng.choice(idx, len(idx), replace=True)
        try:
            aucs.append(roc_auc_score(y[b], p[b]))
        except ValueError:
            pass

    auc = roc_auc_score(y, p)
    lo, hi = np.percentile(aucs, 2.5), np.percentile(aucs, 97.5)
    return auc, lo, hi


def delong_type_test(y, p1, p2):
    """
    Simplified paired ROC comparison.
    Returns:
        delta_auc, p_value
    """
    y = np.asarray(y)
    p1 = np.asarray(p1)
    p2 = np.asarray(p2)

    m = int(np.sum(y))
    n = len(y) - m

    if m == 0 or n == 0:
        return np.nan, np.nan

    a1 = roc_auc_score(y, p1)
    a2 = roc_auc_score(y, p2)

    var = (a1 * (1 - a1) / max(n, 1)) + (a2 * (1 - a2) / max(n, 1))
    z = (a1 - a2) / np.sqrt(var + 1e-12)
    p = 2 * (1 - norm.cdf(abs(z)))
    return a1 - a2, p


def run_lgb_oof_calibrated(X, y, n_splits=5):
    """
    Leakage-free OOF LightGBM with isotonic calibration.
    Returns calibrated OOF predictions.
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    params = dict(
        objective="binary",
        metric="auc",
        learning_rate=0.05,
        num_leaves=31,
        n_estimators=500,
        class_weight="balanced",
        random_state=42,
    )

    oof_pred = np.zeros(len(y))

    for tr_idx, te_idx in skf.split(X, y):
        model = lgb.LGBMClassifier(**params)
        model.fit(X.iloc[tr_idx], y[tr_idx])
        oof_pred[te_idx] = model.predict_proba(X.iloc[te_idx])[:, 1]

    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(oof_pred, y)
    p_cal = iso.transform(oof_pred)

    return p_cal


# =============================================================================
# Table 3: Ablation study
# =============================================================================
table3_records = []

for tag, ycol in [
    ("3M", "3-Month Survival (0=Death, 1=Alive)"),
    ("6M", "6-Month Survival (0=Death, 1=Alive)"),
    ("12M", "12-Month Survival (0=Death, 1=Alive)")
]:
    sub = df.dropna(subset=FEATURES_FULL + [ycol]).copy()
    y = sub[ycol].astype(int).values

    p_full = run_lgb_oof_calibrated(sub[FEATURES_FULL], y)
    p_no = run_lgb_oof_calibrated(sub[FEATURES_NO_ESCC], y)

    auc_full, auc_full_lo, auc_full_hi = auc_ci_bootstrap(y, p_full)
    auc_no, auc_no_lo, auc_no_hi = auc_ci_bootstrap(y, p_no)

    brier_full = brier_score_loss(y, p_full)
    brier_no = brier_score_loss(y, p_no)

    delta_auc, p_delong = delong_type_test(y, p_full, p_no)

    table3_records.append({
        "Horizon": tag,
        "n": len(sub),

        "AUC_Full": auc_full,
        "AUC_Full_95CI_low": auc_full_lo,
        "AUC_Full_95CI_high": auc_full_hi,

        "AUC_No_ESCC": auc_no,
        "AUC_No_ESCC_95CI_low": auc_no_lo,
        "AUC_No_ESCC_95CI_high": auc_no_hi,

        "Delta_AUC": auc_full - auc_no,
        "p_DeLong_type": p_delong,

        "Brier_Full": brier_full,
        "Brier_No_ESCC": brier_no,
        "Delta_Brier": brier_full - brier_no,
    })

table3_df = pd.DataFrame(table3_records)
print("\n=== Table 3: Ablation ===")
print(table3_df)

table3_df.to_excel("Table3_ablation_177_final.xlsx", index=False)
print("Saved: Table3_ablation_177_final.xlsx")


# =============================================================================
# Table 4: Clinical vs Expert vs CNN
# =============================================================================
table4_records = []

for tag, ycol in [
    ("3M", "3-Month Survival (0=Death, 1=Alive)"),
    ("6M", "6-Month Survival (0=Death, 1=Alive)"),
    ("12M", "12-Month Survival (0=Death, 1=Alive)")
]:
    sub = df.dropna(subset=FEATURES_CLINICAL + ["ESCC_expert", "ESCC_cnn", ycol]).copy()
    y = sub[ycol].astype(int).values

    p_clin = run_lgb_oof_calibrated(sub[FEATURES_CLINICAL], y)
    p_exp = run_lgb_oof_calibrated(sub[FEATURES_EXPERT], y)
    p_cnn = run_lgb_oof_calibrated(sub[FEATURES_CNN], y)

    auc_clin, auc_clin_lo, auc_clin_hi = auc_ci_bootstrap(y, p_clin)
    auc_exp, auc_exp_lo, auc_exp_hi = auc_ci_bootstrap(y, p_exp)
    auc_cnn, auc_cnn_lo, auc_cnn_hi = auc_ci_bootstrap(y, p_cnn)

    brier_clin = brier_score_loss(y, p_clin)
    brier_exp = brier_score_loss(y, p_exp)
    brier_cnn = brier_score_loss(y, p_cnn)

    delta_auc_cnn_exp, p_cnn_vs_exp = delong_type_test(y, p_cnn, p_exp)

    table4_records.append({
        "Horizon": tag,
        "n": len(sub),

        "AUC_Clinical": auc_clin,
        "AUC_Clinical_95CI_low": auc_clin_lo,
        "AUC_Clinical_95CI_high": auc_clin_hi,

        "AUC_Expert": auc_exp,
        "AUC_Expert_95CI_low": auc_exp_lo,
        "AUC_Expert_95CI_high": auc_exp_hi,

        "AUC_CNN": auc_cnn,
        "AUC_CNN_95CI_low": auc_cnn_lo,
        "AUC_CNN_95CI_high": auc_cnn_hi,

        "Brier_Clinical": brier_clin,
        "Brier_Expert": brier_exp,
        "Brier_CNN": brier_cnn,

        "Delta_AUC_CNN_minus_Expert": auc_cnn - auc_exp,
        "Delta_Brier_CNN_minus_Expert": brier_cnn - brier_exp,
        "p_DeLong_type_CNN_vs_Expert": p_cnn_vs_exp,
    })

table4_df = pd.DataFrame(table4_records)
print("\n=== Table 4: Expert vs CNN ===")
print(table4_df)

table4_df.to_excel("Table4_expert_vs_cnn_177_final.xlsx", index=False)
print("Saved: Table4_expert_vs_cnn_177_final.xlsx")


print("\nAll analyses completed successfully.")


ストリーミング出力は最後の 5000 行に切り捨てられました。
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain